<a href="https://colab.research.google.com/github/Jakeer437/Impact-of-Renewable-electricity/blob/main/renewable_impact_on_Electricity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Impact of Renewable Electricity Share on CO2 Intensity
# Dataset  : Ember Global Electricity Dataset
# Target   : CO2 Intensity (gCO2/kWh)

In [2]:
# =============================================================================
# SECTION 1: IMPORTS
# =============================================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kstest
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi"     : 150,
    "axes.titlesize" : 13,
    "axes.labelsize" : 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})
plt.style.use("seaborn-v0_8-whitegrid")

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


In [3]:
# =============================================================================
# SECTION 2: CONFIGURATION
# =============================================================================

INPUT_FILE      = "/content/ember_electricity_data.csv"
MODEL_INPUT_CSV = "ember_model_input.csv"
PLOTS_DIR       = "plots"

os.makedirs(PLOTS_DIR, exist_ok=True)

# MLP hyper-parameters
HIDDEN_DIMS   = [256, 128, 64]
DROPOUT_RATE  = 0.3
LEARNING_RATE = 1e-3
EPOCHS        = 200
BATCH_SIZE    = 32
PATIENCE      = 15
VIF_THRESHOLD = 10.0    # drop features iteratively until all VIF <= this value


def save_plot(filename):
    """Save current figure to PLOTS_DIR and close it."""
    plt.savefig(os.path.join(PLOTS_DIR, filename), bbox_inches="tight")
    plt.close()

In [4]:
# =============================================================================
# SECTION 3: LOAD RAW DATA
# =============================================================================

print("Loading raw data ...")
df_raw = pd.read_csv(INPUT_FILE, low_memory=False)
df_raw["Date"] = pd.to_datetime(df_raw["Date"], dayfirst=True)

print(f"Raw shape  : {df_raw.shape}")
print(f"Date range : {df_raw['Date'].min().date()} to {df_raw['Date'].max().date()}")

Loading raw data ...
Raw shape  : (509307, 18)
Date range : 1998-01-12 to 2026-01-01


In [5]:
# =============================================================================
# SECTION 4: FILTER - KEEP ONLY RELEVANT VARIABLES
# =============================================================================

# Restrict to country-level rows only; removes continental/regional aggregates
df_raw = df_raw[df_raw["Area type"] == "Country or economy"].copy()

# Define all (Category, Subcategory, Variable, Unit) combos to retain
relevant_with_unit = [
    ("Power sector emissions",  "CO2 intensity",               "CO2 intensity",               "gCO2/kWh"),
    ("Electricity generation",  "Fuel",                        "Coal",                        "%"),
    ("Electricity generation",  "Fuel",                        "Gas",                         "%"),
    ("Electricity generation",  "Fuel",                        "Wind",                        "%"),
    ("Electricity generation",  "Fuel",                        "Solar",                       "%"),
    ("Electricity generation",  "Fuel",                        "Hydro",                       "%"),
    ("Electricity generation",  "Fuel",                        "Nuclear",                     "%"),
    ("Electricity generation",  "Fuel",                        "Bioenergy",                   "%"),
    ("Electricity generation",  "Fuel",                        "Other Fossil",                "%"),
    ("Electricity generation",  "Fuel",                        "Other Renewables",            "%"),
    ("Electricity generation",  "Aggregate fuel",              "Renewables",                  "%"),
    ("Electricity generation",  "Aggregate fuel",              "Wind and Solar",              "%"),
    ("Electricity generation",  "Aggregate fuel",              "Clean",                       "%"),
    ("Electricity generation",  "Aggregate fuel",              "Fossil",                      "%"),
    ("Electricity demand",      "Demand",                      "Demand",                      "TWh"),
    ("Electricity imports",     "Electricity imports",         "Net Imports",                 "TWh"),
]
relevant_no_unit = [
    ("Electricity prices", "Day-ahead electricity price", "Day-ahead electricity price"),
]

rel_df_u  = pd.DataFrame(relevant_with_unit,
                          columns=["Category", "Subcategory", "Variable", "Unit"])
rel_df_nu = pd.DataFrame(relevant_no_unit,
                          columns=["Category", "Subcategory", "Variable"])

df_long = pd.concat([
    df_raw.merge(rel_df_u,  on=["Category", "Subcategory", "Variable", "Unit"], how="inner"),
    df_raw.merge(rel_df_nu, on=["Category", "Subcategory", "Variable"],         how="inner"),
], ignore_index=True).drop_duplicates()

print(f"\nFiltered long-format shape : {df_long.shape}")
print(f"Unique countries           : {df_long['Area'].nunique()}")


Filtered long-format shape : (147952, 18)
Unique countries           : 88


In [6]:
# =============================================================================
# SECTION 5: PIVOT LONG TO WIDE (one row per country x date)
# =============================================================================

df_long["var_label"] = (df_long["Variable"].str.strip()
                        + " (" + df_long["Unit"].str.strip() + ")")

id_cols = ["Area", "ISO 3 code", "Date", "Continent",
           "Ember region", "EU", "OECD", "G20", "G7", "ASEAN"]

df_wide = (df_long
           .pivot_table(index=id_cols, columns="var_label",
                        values="Value", aggfunc="first")
           .reset_index())
df_wide.columns.name = None
df_wide.rename(columns={"CO2 intensity (gCO2/kWh)": "CO2_intensity"}, inplace=True)
df_wide.dropna(subset=["CO2_intensity"], inplace=True)
df_wide.sort_values(["Area", "Date"], inplace=True)
df_wide.reset_index(drop=True, inplace=True)

print(f"\nWide-format shape : {df_wide.shape}")
print(f"Countries         : {df_wide['Area'].nunique()}")
print(f"Date range        : {df_wide['Date'].min().date()} to {df_wide['Date'].max().date()}")


Wide-format shape : (10615, 27)
Countries         : 88
Date range        : 1998-01-12 to 2025-01-12


In [7]:
# =============================================================================
# SECTION 6: DATA CLEANING AND NULL HANDLING
# =============================================================================

target_col   = "CO2_intensity"
feature_cols = [c for c in df_wide.columns if c not in id_cols + [target_col]]

null_pct = (df_wide[feature_cols].isnull().mean() * 100).round(2)
print(f"\nNull percentage per feature:\n{null_pct.sort_values(ascending=False).to_string()}")

# Drop columns with more than 50% missing values
drop_cols = null_pct[null_pct > 50].index.tolist()
if drop_cols:
    print(f"\nDropping columns with >50% nulls : {drop_cols}")
    df_wide.drop(columns=drop_cols, inplace=True)
    feature_cols = [c for c in feature_cols if c not in drop_cols]

# Forward-fill then backward-fill within each country (preserves time order)
df_wide[feature_cols] = (df_wide
                         .groupby("Area")[feature_cols]
                         .transform(lambda s: s.ffill().bfill()))
df_wide[feature_cols] = df_wide[feature_cols].fillna(df_wide[feature_cols].median())

df_wide.drop_duplicates(inplace=True)
df_wide.reset_index(drop=True, inplace=True)

# Winsorise target at 1st and 99th percentile to remove data entry errors
q01 = df_wide[target_col].quantile(0.01)
q99 = df_wide[target_col].quantile(0.99)
df_wide[target_col] = df_wide[target_col].clip(lower=q01, upper=q99)

print(f"\nAfter cleaning shape        : {df_wide.shape}")
print(f"Remaining nulls in features : {df_wide[feature_cols].isnull().sum().sum()}")


Null percentage per feature:
Day-ahead electricity price (EUR/MWh)    70.79
Other Renewables (%)                     61.98
Nuclear (%)                              57.30
Bioenergy (%)                            23.44
Coal (%)                                 20.82
Solar (%)                                16.32
Other Fossil (%)                         15.06
Wind (%)                                 12.61
Gas (%)                                   9.25
Hydro (%)                                 8.89
Wind and Solar (%)                        6.95
Clean (%)                                 3.88
Renewables (%)                            3.88
Fossil (%)                                0.78
Demand (TWh)                              0.00
Net Imports (TWh)                         0.00

Dropping columns with >50% nulls : ['Day-ahead electricity price (EUR/MWh)', 'Nuclear (%)', 'Other Renewables (%)']

After cleaning shape        : (10615, 24)
Remaining nulls in features : 0


In [8]:
# =============================================================================
# SECTION 7: FEATURE ENGINEERING
# =============================================================================

def get_col(df, col, default=0.0):
    """Return column if it exists, else a zero series."""
    return df[col] if col in df.columns else pd.Series(default, index=df.index)


# Fossil-to-Clean Ratio clipped at p99 to prevent 1e8 outliers distorting gradients
df_wide["fossil_clean_ratio"] = (get_col(df_wide, "Fossil (%)")
                                  / (get_col(df_wide, "Clean (%)") + 1e-6))
df_wide["fossil_clean_ratio"] = df_wide["fossil_clean_ratio"].clip(
    upper=df_wide["fossil_clean_ratio"].quantile(0.99))

# Variable Renewable Share: Wind + Solar (captures intermittent generation)
df_wide["variable_renewable_share"] = (get_col(df_wide, "Wind (%)")
                                        + get_col(df_wide, "Solar (%)"))

# Dispatchable Clean Share: Hydro + Nuclear + Bioenergy (stable zero-carbon sources)
df_wide["dispatchable_clean_share"] = (get_col(df_wide, "Hydro (%)")
                                        + get_col(df_wide, "Nuclear (%)")
                                        + get_col(df_wide, "Bioenergy (%)"))

# Net Import Dependency: net imports as fraction of total demand
demand      = get_col(df_wide, "Demand (TWh)")
net_imports = get_col(df_wide, "Net Imports (TWh)")
df_wide["import_dependency"] = net_imports / (demand + 1e-6)

# Renewables Squared: quadratic term to test nonlinear threshold effects (RQ3)
renewables = get_col(df_wide, "Renewables (%)")
df_wide["renewables_squared"] = renewables ** 2

# Log Demand: reduces leverage of large-economy demand outliers
df_wide["log_demand"] = np.log1p(demand.clip(lower=0))

engineered_cols = [
    "fossil_clean_ratio", "variable_renewable_share", "dispatchable_clean_share",
    "import_dependency",  "renewables_squared",       "log_demand",
]
feature_cols = feature_cols + engineered_cols
print(f"\nFeature columns after engineering ({len(feature_cols)}):\n{feature_cols}")


Feature columns after engineering (19):
['Bioenergy (%)', 'Clean (%)', 'Coal (%)', 'Demand (TWh)', 'Fossil (%)', 'Gas (%)', 'Hydro (%)', 'Net Imports (TWh)', 'Other Fossil (%)', 'Renewables (%)', 'Solar (%)', 'Wind (%)', 'Wind and Solar (%)', 'fossil_clean_ratio', 'variable_renewable_share', 'dispatchable_clean_share', 'import_dependency', 'renewables_squared', 'log_demand']


In [9]:
# =============================================================================
# SECTION 8: STATISTICAL ANALYSIS
# =============================================================================

print("\n" + "=" * 70)
print("STATISTICAL ANALYSIS")
print("=" * 70)

analysis_cols = [target_col] + feature_cols

# -- 8.1 Descriptive Statistics --
print("\n--- 8.1 Descriptive Statistics ---")
desc = df_wide[analysis_cols].describe().T
desc["skewness"] = df_wide[analysis_cols].skew()
desc["kurtosis"] = df_wide[analysis_cols].kurtosis()
print(desc.round(3).to_string())

# -- 8.2 Pearson Correlation with Target --
print("\n--- 8.2 Pearson Correlation with CO2 Intensity ---")
corr_target = (df_wide[feature_cols]
               .corrwith(df_wide[target_col])
               .sort_values(key=abs, ascending=False))
print(corr_target.round(4).to_string())

# -- 8.3 KS Normality Test --
print("\n--- 8.3 Kolmogorov-Smirnov Normality Test ---")
for col in analysis_cols[:8]:
    vals = df_wide[col].dropna()
    std  = (vals - vals.mean()) / (vals.std() + 1e-9)
    ks, p = kstest(std, "norm")
    print(f"  {col:<48}  KS={ks:.4f}  p={p:.4f}  => "
          f"{'NOT normal' if p < 0.05 else 'Normal'}")

# -- 8.4 Iterative VIF-based Feature Selection --
print(f"\n--- 8.4 Iterative VIF Filtering (threshold = {VIF_THRESHOLD}) ---")

exclude_aggregates = ["Renewables (%)", "Wind and Solar (%)", "Clean (%)", "Fossil (%)"]
vif_cols = [c for c in feature_cols
            if c in df_wide.columns
            and c not in exclude_aggregates
            and df_wide[c].isnull().sum() == 0]

# Pre-filter: drop one of any pair with Pearson r > 0.98 (perfect collinearity)
corr_mat  = df_wide[vif_cols].corr().abs()
upper     = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
drop_high = [c for c in upper.columns if (upper[c] > 0.98).any()]
vif_cols  = [c for c in vif_cols if c not in drop_high]
if drop_high:
    print(f"  Pre-VIF drop (r > 0.98) : {drop_high}")

# Iterative VIF dropping
dropped_vif = []
while True:
    mat      = df_wide[vif_cols].values.astype(float)
    vif_vals = [variance_inflation_factor(mat, i) for i in range(len(vif_cols))]
    vif_df   = pd.DataFrame({"Feature": vif_cols, "VIF": vif_vals})
    max_row  = vif_df.loc[vif_df["VIF"].idxmax()]
    if max_row["VIF"] <= VIF_THRESHOLD:
        break
    print(f"  Dropping {max_row['Feature']:<40}  VIF = {max_row['VIF']:.2f}")
    dropped_vif.append(max_row["Feature"])
    vif_cols.remove(max_row["Feature"])

print(f"\nFinal VIF table after iterative dropping:")
mat_final    = df_wide[vif_cols].values.astype(float)
vif_final_df = pd.DataFrame({
    "Feature": vif_cols,
    "VIF"    : [variance_inflation_factor(mat_final, i) for i in range(len(vif_cols))]
}).sort_values("VIF", ascending=False)
print(vif_final_df.round(2).to_string(index=False))

# Only VIF-approved features enter the model
model_features = vif_cols
print(f"\nModel features after VIF filter ({len(model_features)}) : {model_features}")


STATISTICAL ANALYSIS

--- 8.1 Descriptive Statistics ---
                            count         mean           std     min      25%      50%       75%           max  skewness  kurtosis
CO2_intensity             10615.0      423.168  2.491800e+02  21.821  195.220  432.430   612.095  9.687500e+02     0.160    -0.910
Bioenergy (%)             10615.0        2.332  3.646000e+00   0.000    0.390    1.160     2.290  3.750000e+01     3.271    13.844
Clean (%)                 10615.0       46.976  3.011400e+01   0.000   20.990   44.280    73.395  1.000100e+02     0.146    -1.190
Coal (%)                  10615.0       26.560  2.584700e+01   0.000    6.870   18.280    39.905  1.000000e+02     1.184     0.558
Demand (TWh)              10615.0       29.252  9.264700e+01   0.040    1.460    4.780    17.470  1.012800e+03     5.565    35.186
Fossil (%)                10615.0       53.962  3.090900e+01   0.000   26.605   56.820    80.520  1.000000e+02    -0.142    -1.225
Gas (%)                  

In [10]:
# =============================================================================
# SECTION 9: VISUALIZATIONS (15 total, saved to /plots)
# =============================================================================

print("\nGenerating visualizations ...")
df_wide["Year"] = df_wide["Date"].dt.year


# -- PLOT 1: Target Variable Distribution (Histogram + KDE) --
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("CO2 Intensity (Target Variable) Distribution",
             fontsize=14, fontweight="bold")

axes[0].hist(df_wide[target_col], bins=60, color="#2c7bb6",
             edgecolor="white", alpha=0.85)
axes[0].axvline(df_wide[target_col].mean(),   color="crimson",    linestyle="--",
                linewidth=1.8, label=f"Mean   = {df_wide[target_col].mean():.0f}")
axes[0].axvline(df_wide[target_col].median(), color="darkorange", linestyle="--",
                linewidth=1.8, label=f"Median = {df_wide[target_col].median():.0f}")
axes[0].set_xlabel("CO2 Intensity (gCO2/kWh)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Histogram")
axes[0].legend(fontsize=9)

df_wide[target_col].plot(kind="kde", ax=axes[1], color="#2c7bb6", linewidth=2.5)
axes[1].axvline(df_wide[target_col].mean(),   color="crimson",    linestyle="--", linewidth=1.8)
axes[1].axvline(df_wide[target_col].median(), color="darkorange", linestyle="--", linewidth=1.8)
axes[1].fill_between(axes[1].lines[0].get_xdata(),
                      axes[1].lines[0].get_ydata(), alpha=0.15, color="#2c7bb6")
axes[1].set_xlabel("CO2 Intensity (gCO2/kWh)")
axes[1].set_title("KDE Density Curve")

plt.tight_layout()
save_plot("plot_01_target_distribution.png")


# -- PLOT 2: Feature Distributions Grid --
plot2_features = [c for c in model_features if c in df_wide.columns]
n_cols = 4
n_rows = int(np.ceil(len(plot2_features) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(plot2_features):
    axes[i].hist(df_wide[col].dropna(), bins=40, color="#4575b4",
                 edgecolor="white", alpha=0.8)
    axes[i].set_title(col, fontsize=9, fontweight="bold")
    axes[i].set_ylabel("Count")

for j in range(len(plot2_features), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribution of All Model Features",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
save_plot("plot_02_feature_distributions.png")


# -- PLOT 3: Correlation Heatmap (Model Features + Target) --
heatmap_cols  = [target_col] + model_features
corr_mat_plot = df_wide[heatmap_cols].corr()
mask          = np.triu(np.ones_like(corr_mat_plot, dtype=bool))

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr_mat_plot, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, linewidths=0.3,
            ax=ax, annot_kws={"size": 8})
ax.set_title("Correlation Heatmap: Model Features and CO2 Intensity",
             fontsize=13, fontweight="bold")
plt.tight_layout()
save_plot("plot_03_correlation_heatmap.png")


# -- PLOT 4: Global CO2 Intensity vs Renewables Share Over Time (Dual Axis) --
trend = (df_wide
         .groupby("Year")
         .agg(CO2_mean=("CO2_intensity",  "mean"),
              RE_mean  =("Renewables (%)", "mean"))
         .reset_index()
         .dropna())

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.plot(trend["Year"], trend["CO2_mean"], color="#d73027",
         linewidth=2.5, marker="o", markersize=5, label="Mean CO2 Intensity")
ax1.fill_between(trend["Year"], trend["CO2_mean"], alpha=0.1, color="#d73027")
ax2.plot(trend["Year"], trend["RE_mean"], color="#4575b4",
         linewidth=2.5, marker="s", markersize=5, linestyle="--",
         label="Mean Renewables %")
ax2.fill_between(trend["Year"], trend["RE_mean"], alpha=0.1, color="#4575b4")

ax1.set_xlabel("Year")
ax1.set_ylabel("Mean CO2 Intensity (gCO2/kWh)", color="#d73027")
ax2.set_ylabel("Mean Renewables Share (%)", color="#4575b4")
ax1.tick_params(axis="y", labelcolor="#d73027")
ax2.tick_params(axis="y", labelcolor="#4575b4")
ax1.set_title("Global CO2 Intensity vs Renewables Share Over Time",
               fontsize=13, fontweight="bold")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10, loc="upper right")
plt.tight_layout()
save_plot("plot_04_global_co2_renewable_trend.png")


# -- PLOT 5: CO2 Intensity Trend by Country Group (G7 / OECD / G20 / All) --
g7_yr   = df_wide[df_wide["G7"]   == 1].groupby("Year")["CO2_intensity"].mean().rename("G7")
oecd_yr = df_wide[df_wide["OECD"] == 1].groupby("Year")["CO2_intensity"].mean().rename("OECD")
g20_yr  = df_wide[df_wide["G20"]  == 1].groupby("Year")["CO2_intensity"].mean().rename("G20")
all_yr  = df_wide.groupby("Year")["CO2_intensity"].mean().rename("All Countries")

yearly_grp = pd.concat([g7_yr, oecd_yr, g20_yr, all_yr], axis=1).reset_index()

fig, ax = plt.subplots(figsize=(13, 6))
markers = {"G7": "o", "OECD": "s", "G20": "^", "All Countries": "D"}
styles  = {"G7": "-", "OECD": "-", "G20": "-", "All Countries": "--"}
grp_colors = {"G7": "#d73027", "OECD": "#fc8d59", "G20": "#4575b4", "All Countries": "#333333"}

for col in ["G7", "OECD", "G20", "All Countries"]:
    ax.plot(yearly_grp["Year"], yearly_grp[col],
            marker=markers[col], linestyle=styles[col],
            linewidth=2, markersize=5, label=col, color=grp_colors[col])

ax.set_xlabel("Year")
ax.set_ylabel("Mean CO2 Intensity (gCO2/kWh)")
ax.set_title("Mean CO2 Intensity Over Time by Country Group",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
save_plot("plot_05_co2_trend_by_group.png")


# -- PLOT 6: Top 25 Countries by Mean CO2 Intensity (Horizontal Bar) --
country_agg = (
    df_wide
    .groupby(["Area", "Continent"])
    .agg(
        CO2_mean         = ("CO2_intensity",    "mean"),
        Fossil_mean      = ("Fossil (%)",       "mean"),
        Coal_mean        = ("Coal (%)",         "mean"),
        Demand_mean      = ("Demand (TWh)",     "mean"),
        Hydro_mean       = ("Hydro (%)",        "mean"),
        Wind_mean        = ("Wind (%)",         "mean"),
        Solar_mean       = ("Solar (%)",        "mean"),
        Gas_mean         = ("Gas (%)",          "mean"),
        Bioenergy_mean   = ("Bioenergy (%)",    "mean"),
        OtherFossil_mean = ("Other Fossil (%)", "mean"),
    )
    .reset_index()
    .sort_values("CO2_mean", ascending=False)
)

top25      = country_agg.head(25).sort_values("CO2_mean", ascending=True)
conts      = top25["Continent"].unique()
cp         = dict(zip(conts, sns.color_palette("tab10", len(conts))))
bar_colors = top25["Continent"].map(cp)

fig, ax = plt.subplots(figsize=(13, 10))
bars = ax.barh(top25["Area"], top25["CO2_mean"],
               color=bar_colors, edgecolor="white")
for bar, val in zip(bars, top25["CO2_mean"]):
    ax.text(bar.get_width() + 4, bar.get_y() + bar.get_height() / 2,
            f"{val:.0f}", va="center", ha="left", fontsize=8)

handles = [plt.Rectangle((0, 0), 1, 1, color=cp[c]) for c in conts]
ax.legend(handles, conts, title="Continent", fontsize=9)
ax.set_xlabel("Mean CO2 Intensity (gCO2/kWh)")
ax.set_title("Top 25 Countries by Mean CO2 Intensity",
             fontsize=13, fontweight="bold")
ax.set_xlim(0, top25["CO2_mean"].max() * 1.15)
plt.tight_layout()
save_plot("plot_06_top25_countries_co2.png")


# -- PLOT 7: Generation Mix of Top 12 Highest CO2 Countries --
top12   = country_agg.head(12)["Area"].tolist()
fuel_map = {
    "Coal (%)": "Coal", "Gas (%)": "Gas", "Other Fossil (%)": "Other Fossil",
    "Hydro (%)": "Hydro", "Wind (%)": "Wind",
    "Solar (%)": "Solar", "Bioenergy (%)": "Bioenergy",
}
mix_cols = [c for c in fuel_map if c in df_wide.columns]
mix_data = (df_wide[df_wide["Area"].isin(top12)]
            .groupby("Area")[mix_cols]
            .mean()
            .reindex(top12))
mix_data.columns = [fuel_map[c] for c in mix_data.columns]

fuel_colors = {
    "Coal": "#d73027", "Gas": "#fc8d59", "Other Fossil": "#fee090",
    "Hydro": "#4575b4", "Wind": "#74add1", "Solar": "#f4d03f", "Bioenergy": "#91cf60",
}

fig, ax = plt.subplots(figsize=(14, 7))
bottom = np.zeros(len(mix_data))
for fuel in mix_data.columns:
    ax.bar(mix_data.index, mix_data[fuel].values, bottom=bottom,
           label=fuel, color=fuel_colors.get(fuel, "#aaa"),
           edgecolor="white", linewidth=0.4)
    bottom += mix_data[fuel].values

co2_vals = country_agg.set_index("Area").loc[top12, "CO2_mean"].values
ax2 = ax.twinx()
ax2.plot(range(len(top12)), co2_vals, color="black",
         linewidth=2.2, marker="D", markersize=7, label="CO2 Intensity")
ax2.set_ylabel("Mean CO2 Intensity (gCO2/kWh)", fontsize=10)
ax2.legend(loc="upper right", fontsize=9)
ax.set_xticks(range(len(top12)))
ax.set_xticklabels(top12, rotation=30, ha="right")
ax.set_ylabel("Mean Generation Share (%)")
ax.set_title("Generation Mix: Top 12 Highest CO2 Countries",
             fontsize=13, fontweight="bold")
ax.legend(title="Fuel", bbox_to_anchor=(0.01, 1.02),
          loc="upper left", ncol=4, fontsize=9)
plt.tight_layout()
save_plot("plot_07_top12_generation_mix.png")


# -- PLOT 8: CO2 Intensity Trajectory - High Emitters vs Most Improved --
country_year_count = df_wide.groupby("Area")["Year"].nunique()
stable_countries   = country_year_count[country_year_count >= 5].index.tolist()
df_stable          = df_wide[df_wide["Area"].isin(stable_countries)]

yearly_ctry = (df_stable
               .groupby(["Area", "Year"])["CO2_intensity"]
               .mean()
               .reset_index())

first_last = (yearly_ctry
              .sort_values("Year")
              .groupby("Area")
              .agg(first_co2=("CO2_intensity", "first"),
                   last_co2 =("CO2_intensity", "last"))
              .assign(delta=lambda d: d["last_co2"] - d["first_co2"]))

high_co2 = (country_agg[country_agg["Area"].isin(stable_countries)]
            .head(6)["Area"].tolist())
improved = first_last.sort_values("delta").head(6).index.tolist()

pal6 = sns.color_palette("tab10", 6)
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
fig.suptitle("CO2 Trajectory: High Emitters vs Most Improved",
             fontsize=13, fontweight="bold")

for ax, countries, title in zip(
    axes,
    [high_co2, improved],
    ["Top 6 Highest CO2 Countries", "Top 6 Most Improved Countries"]
):
    for i, country in enumerate(countries):
        cdata = yearly_ctry[yearly_ctry["Area"] == country].sort_values("Year")
        ax.plot(cdata["Year"], cdata["CO2_intensity"],
                marker="o", markersize=4, linewidth=2,
                color=pal6[i], label=country)
    ax.set_xlabel("Year")
    ax.set_ylabel("Mean CO2 Intensity (gCO2/kWh)")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)

plt.tight_layout()
save_plot("plot_08_co2_trajectory_countries.png")


# -- PLOT 9: Country x Year CO2 Intensity Heatmap (Top 20) --
top20_countries = country_agg.head(20)["Area"].tolist()
heat_data = (df_wide[df_wide["Area"].isin(top20_countries)]
             .groupby(["Area", "Year"])["CO2_intensity"]
             .mean()
             .unstack("Year"))

fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(heat_data, cmap="RdYlGn_r", linewidths=0.3, ax=ax,
            cbar_kws={"label": "CO2 Intensity (gCO2/kWh)"})
ax.set_title("CO2 Intensity by Country and Year (Top 20 Emitters)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("")
plt.xticks(rotation=45)
plt.tight_layout()
save_plot("plot_09_country_year_heatmap.png")


# -- PLOT 10: Fossil Dependency vs CO2 Intensity Bubble Chart --
plot10      = country_agg.dropna(subset=["CO2_mean", "Fossil_mean", "Demand_mean"]).copy()
cont_list   = plot10["Continent"].unique()
cont_colors = dict(zip(cont_list, sns.color_palette("tab10", len(cont_list))))
d_min, d_max = plot10["Demand_mean"].min(), plot10["Demand_mean"].max()
plot10["bsize"] = 30 + (plot10["Demand_mean"] - d_min) / (d_max - d_min + 1e-9) * 770

fig, ax = plt.subplots(figsize=(14, 9))
for cont, grp in plot10.groupby("Continent"):
    ax.scatter(grp["Fossil_mean"], grp["CO2_mean"],
               s=grp["bsize"], alpha=0.65,
               color=cont_colors.get(cont, "grey"),
               edgecolors="white", linewidth=0.5, label=cont)

for _, row in plot10.nlargest(20, "Demand_mean").iterrows():
    ax.annotate(row["Area"], (row["Fossil_mean"], row["CO2_mean"]),
                fontsize=7.5, ha="center", va="bottom",
                xytext=(0, 5), textcoords="offset points")

med_f, med_c = plot10["Fossil_mean"].median(), plot10["CO2_mean"].median()
ax.axvline(med_f, color="grey", linestyle="--", linewidth=1,
           label=f"Median Fossil = {med_f:.0f}%")
ax.axhline(med_c, color="grey", linestyle=":",  linewidth=1,
           label=f"Median CO2 = {med_c:.0f}")
ax.set_xlabel("Mean Fossil Share (%)")
ax.set_ylabel("Mean CO2 Intensity (gCO2/kWh)")
ax.set_title("Fossil Share vs CO2 Intensity (bubble = demand)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="upper left")
plt.tight_layout()
save_plot("plot_10_fossil_co2_bubble.png")


# -- PLOT 11: Wind + Solar Share Growth Over Time by Continent --
ws_col = ("variable_renewable_share" if "variable_renewable_share" in df_wide.columns
          else "Wind and Solar (%)")
cont_ws = (df_wide
           .groupby(["Year", "Continent"])[ws_col]
           .mean()
           .unstack("Continent")
           .fillna(0))

fig, ax = plt.subplots(figsize=(13, 6))
cont_ws.plot(kind="area", stacked=False, ax=ax, alpha=0.55,
             colormap="tab10", linewidth=2)
ax.set_xlabel("Year")
ax.set_ylabel("Mean Wind + Solar Share (%)")
ax.set_title("Wind + Solar Share Growth Over Time by Continent",
             fontsize=13, fontweight="bold")
ax.legend(title="Continent", fontsize=9, loc="upper left")
plt.tight_layout()
save_plot("plot_11_wind_solar_growth_continent.png")


# -- PLOT 12: Renewables Share vs CO2 Intensity by Continent --
if "Renewables (%)" in df_wide.columns:
    sample_sc = (df_wide
                 .dropna(subset=["Renewables (%)", target_col, "Continent"])
                 .sample(min(6000, len(df_wide)), random_state=RANDOM_STATE))
    conts_sc  = sample_sc["Continent"].unique()
    cp_sc     = dict(zip(conts_sc, sns.color_palette("tab10", len(conts_sc))))

    fig, ax = plt.subplots(figsize=(13, 7))
    for cont, grp in sample_sc.groupby("Continent"):
        ax.scatter(grp["Renewables (%)"], grp[target_col],
                   alpha=0.3, s=15, color=cp_sc.get(cont, "grey"), label=cont)

    z  = np.polyfit(sample_sc["Renewables (%)"], sample_sc[target_col], 2)
    xr = np.linspace(sample_sc["Renewables (%)"].min(),
                     sample_sc["Renewables (%)"].max(), 300)
    ax.plot(xr, np.polyval(z, xr), color="black", linewidth=2.5,
            linestyle="--", label="Global trend (poly-2)")
    ax.set_xlabel("Renewables Share (%)")
    ax.set_ylabel("CO2 Intensity (gCO2/kWh)")
    ax.set_title("Renewables Share vs CO2 Intensity by Continent",
                 fontsize=13, fontweight="bold")
    ax.legend(markerscale=2, fontsize=9)
    plt.tight_layout()
    save_plot("plot_12_renewables_vs_co2.png")



Generating visualizations ...


In [11]:
# =============================================================================
# SECTION 10: PREPARE DATA FOR MODELLING
# =============================================================================

df_model = df_wide[model_features + [target_col]].dropna()
print(f"\nModelling dataset shape     : {df_model.shape}")
print(f"Model features ({len(model_features)}) : {model_features}")

df_model.to_csv(MODEL_INPUT_CSV, index=False)
print(f"Model input saved -> {MODEL_INPUT_CSV}")

X = df_model[model_features].values.astype(np.float32)
y = df_model[target_col].values.astype(np.float32)

# 80 / 10 / 10 split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE)

# Scaler fitted on training data only to prevent leakage into val/test
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"Train : {X_train.shape}  Val : {X_val.shape}  Test : {X_test.shape}")


Modelling dataset shape     : (10615, 12)
Model features (11) : ['Bioenergy (%)', 'Coal (%)', 'Demand (TWh)', 'Gas (%)', 'Hydro (%)', 'Net Imports (TWh)', 'Other Fossil (%)', 'Wind (%)', 'fossil_clean_ratio', 'import_dependency', 'log_demand']
Model input saved -> ember_model_input.csv
Train : (8492, 11)  Val : (1061, 11)  Test : (1062, 11)


In [12]:
# =============================================================================
# SECTION 11: MLP MODEL (PYTORCH)
# =============================================================================

def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

X_tr_t = to_tensor(X_train);  y_tr_t = to_tensor(y_train).unsqueeze(1)
X_va_t = to_tensor(X_val);    y_va_t = to_tensor(y_val).unsqueeze(1)
X_te_t = to_tensor(X_test);   y_te_t = to_tensor(y_test).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t),
                          batch_size=BATCH_SIZE, shuffle=True)


class MLP(nn.Module):
    """
    Feedforward MLP for CO2 intensity regression.
    Linear -> BatchNorm -> ReLU -> Dropout repeated per hidden layer.
    Final layer is a single linear unit (no activation) for regression.
    """
    def __init__(self, input_dim, hidden_dims, dropout_rate):
        super(MLP, self).__init__()
        layers   = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev_dim, h), nn.BatchNorm1d(h),
                       nn.ReLU(), nn.Dropout(dropout_rate)]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


model     = MLP(X_train.shape[1], HIDDEN_DIMS, DROPOUT_RATE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),
                              lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=5, factor=0.5)

print(f"\nMLP Architecture:\n{model}")
print(f"Total parameters : {sum(p.numel() for p in model.parameters()):,}")


train_losses, val_losses = [], []
best_val, patience_count, best_weights = float("inf"), 0, None

print(f"\nTraining MLP (max {EPOCHS} epochs, patience = {PATIENCE}) ...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    batch_mse = []
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()
        batch_mse.append(loss.item())
    train_loss = float(np.mean(batch_mse))

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_va_t), y_va_t).item()

    scheduler.step(val_loss)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val       = val_loss
        patience_count = 0
        best_weights   = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_count += 1

    if epoch % 10 == 0:
        print(f"  Epoch {epoch:>4}/{EPOCHS}  "
              f"Train MSE = {train_loss:.4f}  Val MSE = {val_loss:.4f}")
    if patience_count >= PATIENCE:
        print(f"  Early stopping at epoch {epoch}.")
        break

model.load_state_dict(best_weights)
print(f"Training complete. Best val MSE = {best_val:.4f}")


MLP Architecture:
MLP(
  (network): Sequential(
    (0): Linear(in_features=11, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=64, out_features=1, bias=True)
  )
)
Total parameters : 45,185

Training MLP (max 200 epochs, patience = 15) ...
  Epoch   10/200  Train MSE = 9144.0536  Val MSE = 3493.7048
  Epoch   20/200  Train MSE = 5710.7896  Val MSE = 1431.8130
  Epoch   30/200  Train MSE = 5469.8551  Val MSE = 1092.1117
  Epoc

In [13]:
# =============================================================================
# SECTION 12: EVALUATION AND SCORING
# =============================================================================

model.eval()
with torch.no_grad():
    y_pred_tr = model(X_tr_t).numpy().flatten()
    y_pred_va = model(X_va_t).numpy().flatten()
    y_pred_te = model(X_te_t).numpy().flatten()


def evaluate(y_true, y_pred, split_name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-6))) * 100
    print(f"  {split_name:<12}  MAE={mae:8.4f}  RMSE={rmse:8.4f}  "
          f"R2={r2:.4f}  MAPE={mape:.2f}%")
    return {"Split": split_name, "MAE": mae, "RMSE": rmse, "R2": r2, "MAPE%": mape}


print("\n--- Evaluation Metrics ---")
results = [
    evaluate(y_train, y_pred_tr, "Train"),
    evaluate(y_val,   y_pred_va, "Validation"),
    evaluate(y_test,  y_pred_te, "Test"),
]
print(pd.DataFrame(results).round(4).to_string(index=False))


# -- PLOT 13: Training and Validation Loss Curve --
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(train_losses, linewidth=2, color="#d73027", label="Train MSE")
ax.plot(val_losses,   linewidth=2, color="#4575b4", label="Validation MSE")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("MLP Training and Validation Loss Curve",
             fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
save_plot("plot_13_training_loss.png")


# -- PLOT 14: Actual vs Predicted (Test Set) --
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred_te, alpha=0.3, s=14, color="#2c7bb6")
lo = min(y_test.min(), y_pred_te.min())
hi = max(y_test.max(), y_pred_te.max())
ax.plot([lo, hi], [lo, hi], "r--", linewidth=2, label="Perfect prediction")
ax.fill_between([lo, hi], [lo - 50, hi - 50], [lo + 50, hi + 50],
                alpha=0.08, color="green", label="+/- 50 gCO2/kWh band")
r2_test = r2_score(y_test, y_pred_te)
ax.set_xlabel("Actual CO2 Intensity (gCO2/kWh)")
ax.set_ylabel("Predicted CO2 Intensity (gCO2/kWh)")
ax.set_title(f"Actual vs Predicted - Test Set  (R2 = {r2_test:.4f})",
             fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
save_plot("plot_14_actual_vs_predicted.png")


--- Evaluation Metrics ---
  Train         MAE= 19.4855  RMSE= 27.7760  R2=0.9876  MAPE=8.71%
  Validation    MAE= 20.8703  RMSE= 29.5016  R2=0.9859  MAPE=9.89%
  Test          MAE= 20.2977  RMSE= 27.8166  R2=0.9876  MAPE=9.12%
     Split     MAE    RMSE     R2  MAPE%
     Train 19.4855 27.7760 0.9876 8.7108
Validation 20.8703 29.5016 0.9859 9.8944
      Test 20.2977 27.8166 0.9876 9.1167


In [14]:
# =============================================================================
# SECTION 13: FEATURE IMPORTANCE (PERMUTATION)
# =============================================================================

print("\nComputing permutation feature importance ...")

model.eval()


def mse_on_test(X_tensor):
    with torch.no_grad():
        return mean_squared_error(y_test, model(X_tensor).numpy().flatten())


baseline_mse = mse_on_test(X_te_t)
imp_rows     = []

for i, feat in enumerate(model_features):
    Xp       = X_test.copy()
    Xp[:, i] = np.random.permutation(Xp[:, i])
    delta    = mse_on_test(to_tensor(Xp)) - baseline_mse
    imp_rows.append({"Feature": feat, "MSE_increase": max(0.0, delta)})

imp_df = (pd.DataFrame(imp_rows)
          .sort_values("MSE_increase", ascending=False)
          .reset_index(drop=True))

print("\nFeature Importance (permutation, test set):")
print(imp_df.to_string(index=False))


# -- PLOT 15: Feature Importance Bar Chart --
colors_fi = (["crimson"] * min(5, len(imp_df))
             + ["#2c7bb6"] * max(0, len(imp_df) - 5))

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(imp_df["Feature"][::-1].values,
        imp_df["MSE_increase"][::-1].values,
        color=colors_fi[::-1])
ax.set_xlabel("Increase in MSE When Feature is Permuted")
ax.set_title("Permutation Feature Importance - MLP (Test Set)",
             fontsize=13, fontweight="bold")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
save_plot("plot_15_feature_importance.png")


Computing permutation feature importance ...

Feature Importance (permutation, test set):
           Feature  MSE_increase
          Coal (%)  78929.447876
           Gas (%)  19118.643188
         Hydro (%)  14300.829712
  Other Fossil (%)  10069.423462
          Wind (%)   2155.819702
        log_demand   1924.238159
     Bioenergy (%)   1369.732056
 import_dependency    784.810791
 Net Imports (TWh)    691.334595
      Demand (TWh)    648.859741
fossil_clean_ratio    308.875610


In [15]:
# =============================================================================
# SECTION 14: FORECAST - NEXT 5 YEARS CO2 INTENSITY
# =============================================================================

import matplotlib.dates as mdates

def savefig(name):
    plt.savefig(os.path.join(PLOTS_DIR, name), bbox_inches="tight")
    plt.close()

def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

N_FORECAST_YEARS = 5
HIST_YEARS       = 10
TREND_YEARS      = 7

# -- Step 1: Annual global mean CO2 time series --
df_wide["Year"] = df_wide["Date"].dt.year

annual_hist = (
    df_wide.groupby("Year")[target_col]
    .mean()
    .reset_index()
    .sort_values("Year")
    .rename(columns={target_col: "CO2_mean"})
)

last_year      = int(annual_hist["Year"].max())
forecast_years = list(range(last_year + 1, last_year + 1 + N_FORECAST_YEARS))

print(f"Last year in data : {last_year}")
print(f"Forecasting years : {forecast_years}")


# -- Step 2: Fit linear trend on last TREND_YEARS of annual data --
tail_annual      = annual_hist.tail(TREND_YEARS).copy().reset_index(drop=True)
tail_annual["t"] = np.arange(len(tail_annual))
coeffs           = np.polyfit(tail_annual["t"], tail_annual["CO2_mean"], deg=1)
annual_slope     = coeffs[0]

forecast_t    = np.arange(len(tail_annual), len(tail_annual) + N_FORECAST_YEARS)
forecast_vals = np.polyval(coeffs, forecast_t)

# FIX: floor at a physically reasonable minimum (50 gCO2/kWh = near-zero fossil)
# NOT at the observed minimum, which was the current last value and caused clipping.
PHYSICAL_FLOOR   = 50.0
forecast_vals    = np.clip(forecast_vals, PHYSICAL_FLOOR, None)

print(f"\nTrend slope : {annual_slope:+.2f} gCO2/kWh per year")
for yr, val in zip(forecast_years, forecast_vals):
    print(f"  Forecast {yr} : {val:.1f} gCO2/kWh")


# -- Step 3: Per-country MLP forecast for base uncertainty --
pct_feats    = [c for c in model_features if "(%)" in c]
forecast_ref = (
    df_wide[["Area", "Year"] + model_features]
    .dropna(subset=model_features)
    .sort_values(["Area", "Year"])
)

all_country_rows = []
for country, grp in forecast_ref.groupby("Area"):
    for step, yr in enumerate(forecast_years):
        row = {"Area": country, "Year": yr, "step": step + 1}
        for feat in model_features:
            tail = grp[feat].dropna().tail(TREND_YEARS)
            if len(tail) < 2:
                proj = float(tail.iloc[-1]) if len(tail) == 1 else np.nan
            else:
                x    = np.arange(len(tail))
                z    = np.polyfit(x, tail.values, 1)
                proj = float(np.polyval(z, len(tail) + step))
            row[feat] = (np.clip(proj, 0.0, 100.0)
                         if feat in pct_feats else max(proj, 0.0))
        all_country_rows.append(row)

df_country_fore = pd.DataFrame(all_country_rows).dropna(subset=model_features)

model.eval()
with torch.no_grad():
    X_fc = scaler.transform(
        df_country_fore[model_features].values.astype(np.float32)
    )
    df_country_fore["CO2_forecast"] = model(to_tensor(X_fc)).numpy().flatten()

# Base std from year-1 forecast spread; cone widens by sqrt(step) each year
base_std = float(
    df_country_fore[df_country_fore["step"] == 1]["CO2_forecast"]
    .std()
)
base_std = min(base_std, 20.0)    # cap base std at 20 for readability
print(f"Base uncertainty (year 1 std) : {base_std:.1f} gCO2/kWh")

# Uncertainty grows as sqrt of forecast horizon (standard statistical convention)
fore_std_by_step = np.array([
    base_std * np.sqrt(step) for step in range(1, N_FORECAST_YEARS + 1)
])

fore_df = pd.DataFrame({
    "Year"        : forecast_years,
    "CO2_forecast": forecast_vals,
    "CO2_std"     : fore_std_by_step,
})
fore_df["upper"] = fore_df["CO2_forecast"] + fore_df["CO2_std"]
fore_df["lower"] = (fore_df["CO2_forecast"] - fore_df["CO2_std"]).clip(lower=PHYSICAL_FLOOR)

# Save CSV
df_country_fore[["Area", "Year", "CO2_forecast"]].to_csv(
    "ember_co2_forecast_5years.csv", index=False
)


# -- Step 4: Smooth connecting line (monthly interpolation) --
last_val = float(annual_hist["CO2_mean"].iloc[-1])
all_vals = [last_val] + list(forecast_vals)
all_yrs  = [last_year] + forecast_years

interp_dates, interp_vals = [], []
for i in range(len(all_yrs) - 1):
    seg_dates = pd.date_range(
        start=f"{all_yrs[i]}-01-01",
        end  =f"{all_yrs[i+1]}-01-01",
        freq ="MS"
    )
    seg_vals = np.linspace(all_vals[i], all_vals[i + 1], len(seg_dates))
    interp_dates.extend(seg_dates)
    interp_vals.extend(seg_vals)


# -- Plot --
cutoff    = last_year - HIST_YEARS
hist_plot = annual_hist[annual_hist["Year"] >= cutoff].copy()
hist_plot["date_dt"] = pd.to_datetime(hist_plot["Year"].astype(str) + "-01-01")
fore_dates_dt = pd.to_datetime([f"{y}-01-01" for y in forecast_years])

fig, ax = plt.subplots(figsize=(16, 7))

# Historical line
ax.plot(hist_plot["date_dt"], hist_plot["CO2_mean"],
        color="#2c7bb6", linewidth=2.5, marker="o",
        markersize=8, label="Historical (annual global mean)")

for _, row in hist_plot.iterrows():
    ax.annotate(f"{row['CO2_mean']:.0f}",
                (row["date_dt"], row["CO2_mean"]),
                textcoords="offset points", xytext=(0, 11),
                fontsize=8, ha="center", color="#2c7bb6")

# Trend fit line over history
trend_x     = np.arange(len(tail_annual))
trend_dates = pd.to_datetime([
    f"{last_year - TREND_YEARS + 1 + int(t)}-01-01" for t in trend_x
])
ax.plot(trend_dates, np.polyval(coeffs, trend_x),
        color="#2c7bb6", linewidth=1.2, linestyle=":",
        alpha=0.5,
        label=f"Trend fit (last {TREND_YEARS} yrs, slope={annual_slope:+.1f}/yr)")

# Smooth forecast ramp
ax.plot(interp_dates, interp_vals,
        color="#d73027", linewidth=2.5, linestyle="--",
        label="5-Year Forecast")

# Forecast markers
ax.scatter(fore_dates_dt, forecast_vals,
           color="#d73027", s=100, zorder=5)

# Forecast annotations
for yr_dt, val in zip(fore_dates_dt, forecast_vals):
    ax.annotate(f"{val:.0f}",
                (yr_dt, val),
                textcoords="offset points", xytext=(0, 12),
                fontsize=9, ha="center",
                color="#d73027", fontweight="bold")

# Widening uncertainty cone
ax.fill_between(fore_dates_dt,
                fore_df["lower"], fore_df["upper"],
                alpha=0.15, color="#d73027",
                label="Uncertainty (widens with horizon)")

# Forecast start line
ax.axvline(pd.Timestamp(f"{last_year}-01-01"),
           color="grey", linestyle=":", linewidth=1.5,
           label="Forecast start")

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=0)

ax.set_xlabel("Year")
ax.set_ylabel("Mean CO2 Intensity (gCO2/kWh)")
ax.set_title(
    f"Global Mean CO2 Intensity: Last {HIST_YEARS} Years + "
    f"5-Year Forecast ({forecast_years[0]}–{forecast_years[-1]})",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=9, loc="upper right")
plt.tight_layout()
savefig("plot_16_F1_global_co2_forecast_5yr.png")

print(f"\nPlot  -> {PLOTS_DIR}/plot_16_F1_global_co2_forecast_5yr.png")

Last year in data : 2025
Forecasting years : [2026, 2027, 2028, 2029, 2030]

Trend slope : -10.14 gCO2/kWh per year
  Forecast 2026 : 356.7 gCO2/kWh
  Forecast 2027 : 346.5 gCO2/kWh
  Forecast 2028 : 336.4 gCO2/kWh
  Forecast 2029 : 326.2 gCO2/kWh
  Forecast 2030 : 316.1 gCO2/kWh
Base uncertainty (year 1 std) : 20.0 gCO2/kWh

Plot  -> plots/plot_16_F1_global_co2_forecast_5yr.png
